___

___

# Factorización matricial no negativa

Repitamos la tarea de modelado de la lección anterior, pero esta vez usaremos NMF en lugar de LDA.

## Datos

Usaremos artículos extraídos de NPR (Radio Pública Nacional), obtenidos de su sitio web [www.npr.org] (http://www.npr.org)

In [2]:
import pandas as pd

In [3]:
npr = pd.read_csv('../datos/npr.csv')

In [4]:
npr.head()

,Article
0,"In the Washington of 2016, even when the polic..."
1,Donald Trump has used Twitter — his prefe...
2,Donald Trump is unabashedly praising Russian...
3,"Updated at 2:50 p. m. ET, Russian President Vl..."
4,"From photography, illustration and video, to d..."


¡Observe cómo no tenemos el tema (tópico) de los artículos! Usemos NMF para intentar descifrar grupos de artículos.

## Preprocesamiento

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

/opt/anaconda3/envs/curso_nlp/lib/python3.7/site-packages/sklearn/utils/validation.py:37: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  LARGE_SPARSE_SUPPORTED = LooseVersion(scipy_version) >= '0.14.0'


In [6]:
tfidf = TfidfVectorizer(max_df=0.95, min_df=2, stop_words='english')

In [7]:
dtm = tfidf.fit_transform(npr['Article'])

In [8]:
dtm

<11992x54777 sparse matrix of type '<class 'numpy.float64'>'
	with 3033388 stored elements in Compressed Sparse Row format>

## NMF

In [9]:
from sklearn.decomposition import NMF

In [10]:
nmf_model = NMF(n_components=7,random_state=42)

In [11]:
# Esto puede llevar un tiempo, pero nótese como es menor que con LDA
nmf_model.fit(dtm)

NMF(alpha=0.0, beta_loss='frobenius', init=None, l1_ratio=0.0, max_iter=200,
  n_components=7, random_state=42, shuffle=False, solver='cd', tol=0.0001,
  verbose=0)

## Mostrar temas

In [12]:
len(tfidf.get_feature_names())

54777

In [13]:
import random

In [15]:
for i in range(10):
    random_word_id = random.randint(0,54776)
    print(tfidf.get_feature_names()[random_word_id])

gangsta
bogus
djingareyber
germain
jammeh
meir
sprawls
hardball
sugary
euthanize


In [16]:
len(nmf_model.components_)

7

In [17]:
nmf_model.components_

array([[0.00000000e+00, 2.49950821e-01, 0.00000000e+00, ...,
        1.70313822e-03, 2.37544362e-04, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 8.22048918e-02, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       ...,
       [0.00000000e+00, 3.12379960e-02, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [5.89723338e-03, 0.00000000e+00, 1.50186440e-03, ...,
        7.06428924e-04, 5.85500542e-04, 6.89536542e-04],
       [4.01763234e-03, 5.31643833e-02, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00]])

In [18]:
len(nmf_model.components_[0])

54777

In [19]:
single_topic = nmf_model.components_[0]

In [24]:
# Devuelve los índices que ordenarían esta matriz.
single_topic.argsort()

array([    0, 27208, 27206, ..., 36283, 54692, 42993], dtype=int64)

In [20]:
# Palabra menos representativa de este tema
single_topic[0]

0.0

In [21]:
# Palabra más representativa de este tema
single_topic[42993]

2.0050551654185806

In [22]:
# Las 10 palabras principales para este tema:
single_topic.argsort()[-10:]

array([14441, 36310, 53989, 52615, 47218, 53152, 19307, 36283, 54692,
       42993])

In [23]:
top_word_indices = single_topic.argsort()[-10:]

In [24]:
for index in top_word_indices:
    print(tfidf.get_feature_names()[index])

disease
percent
women
virus
study
water
food
people
zika
says


Quizás parezcan temas de enfermedades... Confirmemos usando .transform() en nuestros artículos vectorizados para adjuntar un número de etiqueta. Pero primero, veamos los 10 temas encontrados.

In [26]:
for index,topic in enumerate(nmf_model.components_):
    print(f'TOP 10 PALABRAS POR TEMA #{index}')
    print([tfidf.get_feature_names()[i] for i in topic.argsort()[-10:]])
    print('\n')

TOP 10 PALABRAS POR TEMA #0
['disease', 'percent', 'women', 'virus', 'study', 'water', 'food', 'people', 'zika', 'says']


TOP 10 PALABRAS POR TEMA #1
['election', 'republican', 'obama', 'white', 'house', 'donald', 'campaign', 'said', 'president', 'trump']


TOP 10 PALABRAS POR TEMA #2
['tax', 'plan', 'republicans', 'affordable', 'obamacare', 'coverage', 'medicaid', 'insurance', 'care', 'health']


TOP 10 PALABRAS POR TEMA #3
['isis', 'russia', 'government', 'state', 'attack', 'president', 'reports', 'court', 'said', 'police']


TOP 10 PALABRAS POR TEMA #4
['party', 'delegates', 'vote', 'state', 'democratic', 'hillary', 'campaign', 'voters', 'sanders', 'clinton']


TOP 10 PALABRAS POR TEMA #5
['time', 'song', 'life', 'really', 'know', 'people', 'think', 'just', 'music', 'like']


TOP 10 PALABRAS POR TEMA #6
['devos', 'children', 'college', 'kids', 'teachers', 'student', 'education', 'schools', 'school', 'students']




### Adjuntar etiquetas de temas descubiertos a artículos originales

In [27]:
dtm

<11992x54777 sparse matrix of type '<class 'numpy.float64'>'
	with 3033388 stored elements in Compressed Sparse Row format>

In [28]:
dtm.shape

(11992, 54777)

In [29]:
len(npr)

11992

In [30]:
topic_results = nmf_model.transform(dtm)

In [31]:
topic_results.shape

(11992, 7)

In [32]:
topic_results[0]

array([0.        , 0.12075603, 0.00140297, 0.05919954, 0.01518909,
       0.        , 0.        ])

In [33]:
topic_results[0].round(2)

array([0.  , 0.12, 0.  , 0.06, 0.02, 0.  , 0.  ])

In [34]:
topic_results[0].argmax()

1

Esto significa que nuestro modelo piensa que el primer artículo pertenece al tema #1.

### Combinación con datos originales

In [35]:
npr.head()

,Article
0,"In the Washington of 2016, even when the polic..."
1,Donald Trump has used Twitter — his prefe...
2,Donald Trump is unabashedly praising Russian...
3,"Updated at 2:50 p. m. ET, Russian President Vl..."
4,"From photography, illustration and video, to d..."


In [36]:
topic_results.argmax(axis=1)

array([1, 1, 1, ..., 0, 4, 3])

In [37]:
npr['Topic'] = topic_results.argmax(axis=1)

In [38]:
npr.head(10)

,Article,Topic
0,"In the Washington of 2016, even when the polic...",1
1,Donald Trump has used Twitter — his prefe...,1
2,Donald Trump is unabashedly praising Russian...,1
3,"Updated at 2:50 p. m. ET, Russian President Vl...",3
4,"From photography, illustration and video, to d...",6
5,I did not want to join yoga class. I hated tho...,5
6,With a who has publicly supported the debunk...,0
7,"I was standing by the airport exit, debating w...",0
8,"If movies were trying to be more realistic, pe...",0
9,"Eighteen years ago, on New Year’s Eve, David F...",5


## Terminamos!